# Cargo Hi5 — Branch A Four Independent Modules

Baseline notebook for Cargo runtime. This trains four modules independently: Y1, Y2, Y3, Y4. Branch B is the recommended scientific path; Branch A is kept for comparison.

This notebook clones `MinhBe/GAN_SQLi`, installs the package, runs tests, detects dataset/GPU, runs smoke first, then runs the full Branch A loop if `RUN_FULL=True`.


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/MinhBe/GAN_SQLi.git'
REPO_BRANCH = 'main'
WORK_ROOT = Path('/workspace') if Path('/workspace').exists() else Path.cwd()
REPO_DIR = WORK_ROOT / 'GAN_SQLi'
RUN_ROOT = WORK_ROOT / 'sqlgan_runs'

RUN_TESTS = True
RUN_SMOKE = True
RUN_FULL = True
FORCE_RECLONE = True

MODULES = ['Y1_basic_boolean', 'Y2_boolean_variation', 'Y3_encoded_boolean', 'Y4_obfuscated_boolean']
SURFACE = 'validation'
FULL_MLE_EPOCHS = 12
FULL_D_EPOCHS = 4
FULL_ADV_EPOCHS = 4
FULL_ADV_STEPS = 100
FULL_ROLLOUTS = 4
FULL_BATCH_SIZE = 256
FULL_GENERATE_N = 5000
MAX_LEN = 192

SMOKE_ADV_EPOCHS = 1
SMOKE_OUT = RUN_ROOT / 'branch_a_smoke'
FULL_OUT = RUN_ROOT / 'branch_a_full'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('WORK_ROOT =', WORK_ROOT)
print('REPO_DIR  =', REPO_DIR)
print('RUN_ROOT  =', RUN_ROOT)


In [ ]:
import os, sys, subprocess, shutil, platform, json

def run(cmd, cwd=None, check=True):
    print('\n$ ' + ' '.join(map(str, cmd)))
    p = subprocess.run(list(map(str, cmd)), cwd=cwd, text=True)
    if check and p.returncode != 0:
        raise SystemExit(p.returncode)
    return p.returncode

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
except Exception as e:
    print('Torch import before install failed:', repr(e))


In [ ]:
if FORCE_RECLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR)


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=REPO_DIR)
run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=REPO_DIR)
if RUN_TESTS:
    run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR)


In [ ]:
candidates = [REPO_DIR, REPO_DIR / 'SQLGAN_PostgreSQL_Boolean_Attack_Corpus_V4_1_StaticTrainingCorpus', WORK_ROOT]
DATA_PATH = None
for c in candidates:
    if (c / 'A_generator_attack_corpus').exists() or (c / 'Dataset' / 'A_generator_attack_corpus').exists():
        DATA_PATH = c
        break
if DATA_PATH is None:
    zips = list(WORK_ROOT.rglob('*V4*StaticTrainingCorpus*.zip')) + list(WORK_ROOT.rglob('*SQLGAN*Boolean*.zip'))
    if zips:
        DATA_PATH = zips[0]
if DATA_PATH is None:
    raise FileNotFoundError('Cannot find SQLGAN dataset. Put dataset folder/zip in repo or Cargo workspace.')
print('DATA_PATH =', DATA_PATH)


In [ ]:
audit_json = RUN_ROOT / 'length_audit_branch_a.json'
run([sys.executable, '-m', 'sqlgan_dual.audit_length', '--data', str(DATA_PATH), '--out-json', str(audit_json), '--surface', SURFACE, '--max-len', str(MAX_LEN)], cwd=REPO_DIR)
print(audit_json.read_text()[:4000])


In [ ]:
def train_module_set(out_root, smoke):
    out_root.mkdir(parents=True, exist_ok=True)
    reports = []
    for module_id in MODULES:
        mod_out = out_root / module_id
        cmd = [sys.executable, '-m', 'sqlgan_dual.train_module',
               '--data', str(DATA_PATH), '--out', str(mod_out), '--module-id', module_id,
               '--surface', SURFACE, '--max-len', str(MAX_LEN)]
        if smoke:
            cmd += ['--smoke', '--adv-epochs', str(SMOKE_ADV_EPOCHS)]
        else:
            cmd += ['--mle-epochs', str(FULL_MLE_EPOCHS),
                    '--d-epochs', str(FULL_D_EPOCHS),
                    '--adv-epochs', str(FULL_ADV_EPOCHS),
                    '--adv-steps', str(FULL_ADV_STEPS),
                    '--rollouts', str(FULL_ROLLOUTS),
                    '--batch-size', str(FULL_BATCH_SIZE),
                    '--generate-n', str(FULL_GENERATE_N)]
        run(cmd, cwd=REPO_DIR)
        summary = mod_out / 'module_summary.json'
        if summary.exists():
            reports.append(json.loads(summary.read_text()))
    branch_summary = {'branch': 'A_four_independent_modules', 'smoke': smoke, 'reports': reports}
    (out_root / 'branch_a_summary.json').write_text(json.dumps(branch_summary, ensure_ascii=False, indent=2))
    return branch_summary


In [ ]:
if RUN_SMOKE:
    rep = train_module_set(SMOKE_OUT, smoke=True)
    print(json.dumps({'out': str(SMOKE_OUT), 'modules': len(rep['reports'])}, indent=2))


In [ ]:
if RUN_FULL:
    rep = train_module_set(FULL_OUT, smoke=False)
    print(json.dumps({'out': str(FULL_OUT), 'modules': len(rep['reports'])}, indent=2))


In [ ]:
for p in [SMOKE_OUT / 'branch_a_summary.json', FULL_OUT / 'branch_a_summary.json', RUN_ROOT / 'length_audit_branch_a.json']:
    if p.exists():
        print('\n====', p, '====')
        print(p.read_text()[:8000])
